# 📊 Udemy Courses Analysis
## EDA + Machine Learning Model
**Dataset:** Udemy Courses | **Target:** Predict Number of Subscribers

---
## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Model libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

---
## 2. 📂 Load Data

In [ ]:
df = pd.read_csv('udemy_courses.csv')
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(5)

---
## 3. 🔍 Data Overview

In [ ]:
df.info()

In [ ]:
# Numeric columns statistics
df.describe()

In [ ]:
# Missing values check
print('🔎 Missing Values:')
print(df.isnull().sum())

In [ ]:
# Duplicates check
print(f'🔁 Duplicate rows: {df.duplicated().sum()}')

---
## 4. 🧹 Data Cleaning

In [ ]:
# Remove duplicates
df.drop_duplicates(inplace=True)
print(f'✅ After removing duplicates: {df.shape}')

In [ ]:
# Remove illogical entries (courses with 0 lectures)
df = df[df['num_lectures'] > 0]
print(f'✅ After removing 0-lecture courses: {df.shape}')

In [ ]:
# Drop irrelevant columns
df.drop(['course_id', 'url'], axis=1, inplace=True)

# Parse dates and extract time features
df['published_timestamp'] = pd.to_datetime(df['published_timestamp'])
df['year'] = df['published_timestamp'].dt.year
df['month'] = df['published_timestamp'].dt.month

print(f'✅ Final shape: {df.shape}')
df.info()

---
## 5. 📊 Exploratory Data Analysis (EDA)

### 5.1 Paid vs Free Courses

In [ ]:
sns.countplot(x='is_paid', data=df)
plt.title('Count of Paid vs Free Courses')
plt.xlabel('Is Paid?')
plt.ylabel('Number of Courses')
plt.show()

print(df['is_paid'].value_counts())

In [ ]:
counts = df['is_paid'].value_counts()
labels = ['Paid', 'Free']

plt.figure(figsize=(6, 6))
plt.pie(counts, labels=labels, autopct='%1.2f%%', startangle=140,
        colors=['#ff9999','#66b3ff'], pctdistance=0.85)

centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title('Distribution of Paid vs Not Paid Courses')
plt.axis('equal')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='is_paid', y='num_subscribers', data=df, palette='magma')

plt.text(0, df[df['is_paid']==False]['num_subscribers'].mean() + 500,
         'Free courses are in higher demand',
         fontsize=12, color='darkgreen', fontweight='bold', ha='center')

plt.xticks([0, 1], ['Free Courses', 'Paid Courses'])
plt.title('Subscribers Demand: Free vs Paid', fontsize=15)
plt.xlabel('Type of Course')
plt.ylabel('Average Subscribers')
plt.show()

### 5.2 Subject Analysis

In [ ]:
print(df['subject'].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(y='subject', data=df, order=df['subject'].value_counts().index, palette='viridis')
plt.title('Number of Courses per Subject')
plt.xlabel('Count of Courses')
plt.ylabel('Subject')
plt.show()

In [ ]:
subject_counts = df['subject'].value_counts()
explode_values = [0.2, 0.05, 0.05, 0.05]

plt.figure(figsize=(10, 8))
plt.pie(subject_counts,
        labels=subject_counts.index,
        autopct='%1.1f%%',
        startangle=140,
        explode=explode_values,
        shadow=True,
        colors=plt.cm.Set3.colors)

plt.title('Percentage of Each Subject (Top Subject Highlighted)', fontsize=15, fontweight='bold')
plt.axis('equal')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='subject', y='num_subscribers', data=df, estimator=sum, ci=None, palette='viridis')
plt.title('Total Subscribers per Subject')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x='subject', y='num_subscribers', hue='is_paid', data=df, estimator=sum, ci=None)
plt.title('Total Subscribers per Subject: Free vs. Paid Courses', fontsize=15, fontweight='bold')
plt.xticks(rotation=45)
plt.xlabel('Subject')
plt.ylabel('Total Subscribers')
plt.legend(title='Is Paid?')
plt.show()

### 5.3 Course Level Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='level', order=df.level.value_counts().index)
plt.title('Course Level Distribution')
plt.xlabel('Level')
plt.ylabel('Count')
plt.show()

### 5.4 Price Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], bins=20, kde=True, color='teal')
plt.title('Distribution of Course Prices', fontsize=15, fontweight='bold')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

### 5.5 Relationships Between Features

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='price', y='num_subscribers', data=df, alpha=0.5, color='darkblue')
plt.title('Relationship between Course Price and Number of Subscribers', fontsize=14, fontweight='bold')
plt.xlabel('Price ($)')
plt.ylabel('Number of Subscribers')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(x='num_lectures', y='num_subscribers', data=df,
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title('Impact of Number of Lectures on Student Enrollment', fontsize=14, fontweight='bold')
plt.xlabel('Number of Lectures')
plt.ylabel('Number of Subscribers')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(x='content_duration', y='num_subscribers', data=df,
            scatter_kws={'alpha':0.3, 'color':'teal'}, line_kws={'color':'orange'})
plt.title('Course Duration vs. Number of Subscribers', fontsize=14, fontweight='bold')
plt.xlabel('Content Duration (in Hours)')
plt.ylabel('Number of Subscribers')
plt.show()

### 5.6 Yearly Revenue & Publication Trends

In [ ]:
yearly_revenue = df.groupby('year')['price'].sum()
yearly_revenue.plot(kind='line', marker='o', color='green')
plt.title('Total Course Revenue by Year')
plt.xlabel('Year')
plt.ylabel('Total Price')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='month', data=df, palette='magma')
plt.xticks(ticks=range(0, 12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                                        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.title('Total Courses Published per Month (All Years)', fontsize=14)
plt.show()

### 5.7 Top Courses

In [ ]:
top_5_courses = df.nlargest(5, 'num_subscribers')
plt.figure(figsize=(12, 6))
sns.barplot(x='num_subscribers', y='course_title', data=top_5_courses, palette='coolwarm')
plt.title('Top 5 Most Enrolled Courses', fontsize=15, fontweight='bold')
plt.xlabel('Number of Subscribers')
plt.ylabel('Course Title')
plt.show()

In [ ]:
top_5_reviewed = df.nlargest(5, 'num_reviews')
plt.figure(figsize=(12, 6))
sns.barplot(x='num_reviews', y='course_title', data=top_5_reviewed, palette='plasma')
plt.title('Top 5 Courses by Number of Reviews', fontsize=15, fontweight='bold')
plt.xlabel('Number of Reviews')
plt.ylabel('Course Title')
plt.show()

---
## 6. ➕ Additional EDA (New Additions)

### 6.1 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
numeric_cols = df.select_dtypes(include='number')
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Between Numeric Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 Outlier Detection with Boxplots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['price', 'num_subscribers', 'num_reviews']):
    sns.boxplot(y=col, data=df, ax=ax, color='lightblue')
    ax.set_title(f'Boxplot of {col}', fontweight='bold')
plt.suptitle('Outlier Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. ⚙️ Feature Engineering

In [ ]:
# Review rate: ratio of reviews to subscribers
df['review_rate'] = df['num_reviews'] / (df['num_subscribers'] + 1)

# Average lecture duration in minutes
df['avg_lecture_duration'] = df['content_duration'] / (df['num_lectures'] + 1)

print('✅ New features added:')
print('  - review_rate: num_reviews / (num_subscribers + 1)')
print('  - avg_lecture_duration: content_duration / (num_lectures + 1)')
df[['review_rate', 'avg_lecture_duration']].describe()

---
## 8. 🤖 Machine Learning Model — Random Forest Regression

### 8.1 Preprocessing for Model

In [ ]:
df_model = df.copy()

# Encode categorical columns
le_level = LabelEncoder()
le_subject = LabelEncoder()

df_model['level_encoded'] = le_level.fit_transform(df_model['level'])
df_model['subject_encoded'] = le_subject.fit_transform(df_model['subject'])
df_model['is_paid_int'] = df_model['is_paid'].astype(int)

print('Level encoding:')
for i, label in enumerate(le_level.classes_):
    print(f'  {label} → {i}')

print('\nSubject encoding:')
for i, label in enumerate(le_subject.classes_):
    print(f'  {label} → {i}')

### 8.2 Define Features & Target

In [ ]:
features = [
    'is_paid_int',
    'price',
    'num_reviews',
    'num_lectures',
    'level_encoded',
    'content_duration',
    'subject_encoded',
    'year',
    'month',
    'review_rate',
    'avg_lecture_duration'
]

X = df_model[features]
y = df_model['num_subscribers']

print(f'✅ Features shape: {X.shape}')
print(f'✅ Target shape: {y.shape}')
X.head()

### 8.3 Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'✅ Training set size : {X_train.shape[0]} rows')
print(f'✅ Testing set size  : {X_test.shape[0]} rows')

### 8.4 Train the Model

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print('✅ Model trained successfully!')

### 8.5 Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=' * 40)
print('       📊 Model Evaluation Results')
print('=' * 40)
print(f'  MAE  (Mean Absolute Error) : {mae:,.0f}')
print(f'  RMSE (Root Mean Sq. Error) : {rmse:,.0f}')
print(f'  R²   (R-Squared Score)     : {r2:.4f}')
print('=' * 40)

### 8.6 Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue', edgecolors='k', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Subscribers')
plt.ylabel('Predicted Subscribers')
plt.title('Actual vs Predicted Number of Subscribers', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

### 8.7 Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Feature Importance — Random Forest', fontsize=15, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print('\n📋 Feature Importance Table:')
print(importance_df.to_string(index=False))

---
## 9. ✅ Summary

| Step | Details |
|------|--------|
| Dataset | Udemy Courses (3678 rows, 12 columns) |
| Target | `num_subscribers` (Regression) |
| Model | Random Forest Regressor (100 trees) |
| New Features | `review_rate`, `avg_lecture_duration` |
| Train/Test Split | 80% / 20% |

### Key Insights from EDA:
- 🔹 ~93% of courses are **paid**
- 🔹 **Free courses** attract significantly more subscribers on average
- 🔹 **Web Development** has the highest number of courses and subscribers
- 🔹 **num_reviews** is the strongest predictor of subscriber count